# **Tarjeta de Viajes 2025**

Este notebook carga el archivo "Viajes por empresa.csv" y muestra una tarjeta con el número de viajes realizados por la empresa seleccionada en 2025.

In [1]:
# Section 1: Importar librerías
import pandas as pd
from IPython.display import HTML, display
import ipywidgets as widgets
from datetime import datetime
import os


In [2]:
# Section 2: Cargar dataset "viajes"
# Probar varios nombres comunes y manejar BOM/encoding
candidates = ['Viajes por empresa.csv','Viajes%20por%20empresa.csv','viajes por empresa.csv','Viajes_por_empresa.csv']
path = None
for f in candidates:
    if os.path.exists(f):
        path = f
        break
if path is None:
    raise FileNotFoundError(f"No se encontró 'Viajes por empresa.csv'. Archivos probados: {candidates}")

# Leer CSV
try:
    df = pd.read_csv(path, encoding='utf-8', low_memory=False)
except Exception:
    df = pd.read_csv(path, encoding='latin-1', low_memory=False)

# Limpiar encabezados
df.columns = [c.replace('\ufeff','').strip() for c in df.columns]

display(df.head())


,Partida Aduanera,Descripcion de la Partida Aduanera,Aduana,DUA,Fecha,Cod. Tributario,Exportador,Ciudad Exportador,Dirección Exportador,Estado Exportador,...,Und 1,Qty 2,Und 2,U$ Valor Tot,U$ Valor Unit 1,U$ Valor Unit 2,País Destino,Via,Descripción Comercial,Regimen
0,306179100,"LOS DEMÁS CAMARONES, LANGOSTINOS Y DEMÁS DECÁP...","NOGALES, SONORA",5020636,30/09/25,ASE001115EN2,Ahome Acuicola SA de CV,NaN,N/D,NaN,...,KG,"8,215.60",KG,"62,486.38",7.606,7.606,ESTADOS UNIDOS,CARRETERO,CAMARON DE CULTIVO FRESCO CONGELADO xxx,EXPORTACION DEFINITIVA.
1,306179100,"LOS DEMÁS CAMARONES, LANGOSTINOS Y DEMÁS DECÁP...","NOGALES, SONORA",5020636,30/09/25,ASE001115EN2,Ahome Acuicola SA de CV,NaN,N/D,NaN,...,KG,"7,009.03",KG,"53,309.39",7.606,7.606,ESTADOS UNIDOS,CARRETERO,CAMARON DE CULTIVO FRESCO CONGELADO xxx,EXPORTACION DEFINITIVA.
2,306179100,"LOS DEMÁS CAMARONES, LANGOSTINOS Y DEMÁS DECÁP...","NOGALES, SONORA",5020636,30/09/25,ASE001115EN2,Ahome Acuicola SA de CV,NaN,N/D,NaN,...,KG,504.40,KG,"3,836.42",7.606,7.606,ESTADOS UNIDOS,CARRETERO,CAMARON DE CULTIVO FRESCO CONGELADO xxx,EXPORTACION DEFINITIVA.
3,306179100,"LOS DEMÁS CAMARONES, LANGOSTINOS Y DEMÁS DECÁP...","NOGALES, SONORA",5019751,24/09/25,ASE001115EN2,Ahome Acuicola SA de CV,NaN,N/D,NaN,...,KG,"7,094.30",KG,"54,576.81",7.693,7.693,ESTADOS UNIDOS,CARRETERO,CAMARON DE CULTIVO FRESCO CONGELADO xxx,EXPORTACION DEFINITIVA.
4,306179100,"LOS DEMÁS CAMARONES, LANGOSTINOS Y DEMÁS DECÁP...","NOGALES, SONORA",5019751,22/09/25,ASE001115EN2,Ahome Acuicola SA de CV,NaN,N/D,NaN,...,KG,"1,339.03",KG,"10,036.75",7.496,7.496,ESTADOS UNIDOS,CARRETERO,CAMARON DE CULTIVO FRESCO CONGELADO xxx,EXPORTACION DEFINITIVA.


In [3]:
# Section 3: Preprocesar fechas y filtrar 2025
# Intentar detectar columna de fecha
date_col = None
for c in df.columns:
    if c.lower() in ['fecha','date','fecha_viaje','fecha de viaje','fecha_viaje']:
        date_col = c
        break
# Si no hay columna de fecha, buscar columnas que contengan 'año' o 'year'
if date_col is None:
    for c in df.columns:
        if 'año' in c.lower() or 'year' in c.lower() or 'anio' in c.lower():
            date_col = c
            break

if date_col:
    df['fecha_parsed'] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
else:
    # fallback: intentar parsear la primera columna
    df['fecha_parsed'] = pd.to_datetime(df.iloc[:,0], errors='coerce', dayfirst=True)

# Extraer año
df['anio'] = df['fecha_parsed'].dt.year
# Filtrar 2025
df_2025 = df[df['anio'] == 2025]
print(f"Registros totales: {len(df)}, registros 2025: {len(df_2025)}")


Registros totales: 425, registros 2025: 425


In [4]:
# Section 4: Selector interactivo de empresa
# Detectar columna de empresa
comp_col = None
for c in df.columns:
    if c.lower() in ['empresa','company','empresa_nombre','razon social']:
        comp_col = c
        break
if comp_col is None:
    # fallback: primera columna de texto
    for c in df.columns:
        if df[c].dtype == object:
            comp_col = c
            break

empresas = sorted(df_2025[comp_col].dropna().astype(str).unique())

if not empresas:
    empresas = sorted(df[comp_col].dropna().astype(str).unique())

dropdown = widgets.Dropdown(options=empresas, description='Empresa:', layout=widgets.Layout(width='60%'))
display(dropdown)


Dropdown(description='Empresa:', layout=Layout(width='60%'), options=('ATUNES (DEL GÉNERO "THUNUS"), EXCEPTO L…

In [6]:
# Section 5 & 6: Calcular número de viajes en 2025 y renderizar tarjeta
from IPython.display import HTML

def render_trips_card(empresa):
    empresa_key = str(empresa).strip()
    mask = df_2025[comp_col].astype(str).str.strip().str.lower() == empresa_key.lower()
    subset = df_2025[mask]
    # intentar detectar columna numérica de conteo
    num_col = None
    for k in subset.columns:
        if any(s in k.lower() for s in ['viaje','trip','count','cantidad','numero','num']):
            if pd.api.types.is_numeric_dtype(subset[k]):
                num_col = k
                break
    if num_col:
        count = int(subset[num_col].sum())
    else:
        count = int(len(subset))
    html = f"""
    <div style='border-radius:12px;padding:24px;background:#eef7ef;color:#0b6b4f;display:flex;flex-direction:column;align-items:center;justify-content:center;width:480px;'>
      <div style='font-weight:700;font-size:16px;margin-bottom:8px'>Viajes realizados por <span style='color:#0b6b4f'>{empresa_key}</span> en 2025:</div>
      <div style='font-weight:800;font-size:56px;margin:8px 0;color:#0b6b4f'>{count:,}</div>
      <div style='font-size:14px;color:#2f4f3f'>viajes</div>
    </div>
    """
    display(HTML(html))

out = widgets.Output()
display(out)

def on_change(change):
    with out:
        out.clear_output()
        render_trips_card(change['new'])

if empresas:
    dropdown.observe(on_change, names='value')
    # render inicial
    render_trips_card(empresas[0])
else:
    with out:
        display(HTML("<div>No hay empresas detectadas en los datos de 2025.</div>"))


Output()

In [ ]:
# Section 7: Prueba rápida / assert
assert isinstance(df, pd.DataFrame), 'df debe ser un DataFrame'
assert comp_col in df.columns, f"Columna de empresa no encontrada entre: {df.columns.tolist()}"
if empresas:
    test_count = df_2025[df_2025[comp_col].astype(str).str.strip().str.lower() == empresas[0].lower()].shape[0]
    assert isinstance(test_count, int) and test_count >= 0
print('Checks passed.')


# Instrucciones rápidas

# Instalar dependencias si es necesario:
# pip install pandas ipywidgets

# Ejecutar el notebook con Jupyter Lab o Notebook:
# jupyter lab
# o
# jupyter notebook

# Seleccione una empresa en el dropdown para actualizar la tarjeta.
